# Installing Ollama

In [1]:
!apt-get update -qq
!apt-get install -y zstd

!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 112 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 2s (255 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122579 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-

In [2]:
!nvidia-smi

Thu Sep  3 10:38:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Running Ollama server

In [3]:
%%bash

# Stop any old Ollama processes
pkill -f "ollama serve" 2>/dev/null || true
pkill -f "ollama_supervisor" 2>/dev/null || true

# Keep models loaded indefinitely
export OLLAMA_KEEP_ALIVE=-1
export OLLAMA_HOST=127.0.0.1:11434
export OLLAMA_ORIGINS="*"

# Create supervisor
cat > /tmp/ollama_supervisor.sh <<'EOF'
#!/bin/bash

export OLLAMA_KEEP_ALIVE=-1
export OLLAMA_HOST=127.0.0.1:11434

while true; do
    echo "$(date): Starting Ollama..." >> /tmp/ollama_supervisor.log

    ollama serve >> /tmp/ollama.log 2>&1

    EXIT_CODE=$?

    echo "$(date): Ollama exited with code $EXIT_CODE. Restarting..." \
        >> /tmp/ollama_supervisor.log

    sleep 2
done
EOF

chmod +x /tmp/ollama_supervisor.sh

# Start supervisor completely detached from this shell
nohup /tmp/ollama_supervisor.sh \
    > /tmp/ollama_supervisor.log 2>&1 < /dev/null &

# Wait for Ollama
echo "Waiting for Ollama..."

for i in {1..30}; do
    if curl -sf http://127.0.0.1:11434/api/version > /tmp/ollama_version.json; then
        break
    fi
    sleep 1
done

# Verify
echo
echo "=== Ollama ==="
cat /tmp/ollama_version.json 2>/dev/null || {
    echo "Ollama failed to start"
    tail -50 /tmp/ollama.log
    exit 1
}

echo
echo "=== Process ==="
ps aux | grep '[o]llama'

echo
echo "=== Port ==="
ss -ltnp | grep 11434 || true

# Pull model
ollama pull qwen3:8b

# Show models
ollama list

Waiting for Ollama...

=== Ollama ===
{"version":"0.33.2"}
=== Process ===
root        2228  0.0  0.0   7376  3492 ?        S    10:38   0:00 /bin/bash /tmp/ollama_supervisor.sh
root        2231  2.7  0.2 1821208 38668 ?       Sl   10:38   0:00 ollama serve

=== Port ===
LISTEN 0      4096       127.0.0.1:11434      0.0.0.0:*    users:(("ollama",pid=2231,fd=4))       
NAME        ID              SIZE      MODIFIED               
qwen3:8b    500a1f067a9f    5.2 GB    Less than a second ago    


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling a3de86cd1c13:   0% ▕                  ▏ 122 KB/5.2 GB                  pulling manifest 
pulling a3de86cd1c13:   1% ▕                  ▏  70 MB/5.2 GB                  pulling manifest 
pulling a3de86cd1c13:   3% ▕                  ▏ 136 MB/5.2 GB                  pulling manifest 
pulling a3de86cd1c13:   3% ▕                  ▏ 167 MB/5.2 GB                  pulling manifest 
pulling a3de86cd1c13:   4% ▕                  ▏ 215 MB/5.2 GB                  pulling manifest 
pulling a3de86cd1c13:   5% ▕             

> Checking the server

In [4]:
import requests
import pprint

r = requests.get("http://127.0.0.1:11434/api/tags")
t = requests.get("http://127.0.0.1:11434/api/version")

print(r.status_code)
pprint.pp(r.json())
print(t.status_code)
pprint.pp(t.json())

200
{'models': [{'name': 'qwen3:8b',
             'model': 'qwen3:8b',
             'modified_at': '2026-09-03T10:39:13.200407293Z',
             'size': 5225388164,
             'digest': '500a1f067a9f782620b40bee6f7b0c89e17ae61f686b92c24933e4ca4b2b8b41',
             'details': {'parent_model': '',
                         'format': 'gguf',
                         'family': 'qwen3',
                         'families': ['qwen3'],
                         'parameter_size': '8.2B',
                         'quantization_level': 'Q4_K_M',
                         'context_length': 40960,
                         'embedding_length': 4096},
             'capabilities': ['completion', 'tools', 'thinking']}]}
200
{'version': '0.33.2'}


> Checking Structured Output using ollama

In [5]:
!pip install -q ollama pydantic

In [6]:
from ollama import Client
from pydantic import BaseModel, Field
from pprint import pp

class BISResponse(BaseModel):
    answer: str
    confidence: float = Field(
        ge=0.0,
        le=1.0
    )


client = Client(
    host="http://127.0.0.1:11434"
)


response = client.chat(
    model="qwen3:8b",

    messages=[
        {
            "role": "user",
            "content": "What is BIS?"
        }
    ],

    format=BISResponse.model_json_schema(),

    options={
        "temperature": 0,
    }
)


pp(response["message"]["content"])

('{"answer": "BIS can refer to different organizations depending on the '
 'context. The most prominent one is the **Bank for International Settlements '
 '(BIS)**, which is an international financial institution headquartered in '
 'Basel, Switzerland. It serves as a forum for central banks and monetary '
 'authorities to collaborate on global financial stability, monetary policy, '
 'and regulatory standards. Key functions include:\\n\\n- Facilitating '
 'cooperation among central banks.\\n- Conducting research on global financial '
 'systems.\\n- Setting standards for financial regulations (e.g., Basel '
 'Accords).\\n- Acting as a bank for central banks (e.g., holding reserves for '
 'member institutions).\\n\\nOther possible meanings include:\\n- **Bureau of '
 'Indian Standards (BIS)**: A standards organization in India that sets '
 'quality and safety standards for products.\\n- **British Indian Society '
 "(BIS)**: A historical cultural organization in the UK.\\n\\nIf you're "


In [7]:
result = BISResponse.model_validate_json(
    response["message"]["content"]
)

print(result)
print(result.answer)
print(result.confidence)

answer="BIS can refer to different organizations depending on the context. The most prominent one is the **Bank for International Settlements (BIS)**, which is an international financial institution headquartered in Basel, Switzerland. It serves as a forum for central banks and monetary authorities to collaborate on global financial stability, monetary policy, and regulatory standards. Key functions include:\n\n- Facilitating cooperation among central banks.\n- Conducting research on global financial systems.\n- Setting standards for financial regulations (e.g., Basel Accords).\n- Acting as a bank for central banks (e.g., holding reserves for member institutions).\n\nOther possible meanings include:\n- **Bureau of Indian Standards (BIS)**: A standards organization in India that sets quality and safety standards for products.\n- **British Indian Society (BIS)**: A historical cultural organization in the UK.\n\nIf you're referring to a specific context, please clarify!" confidence=0.95
B

# Installing Cloudflare

In [8]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

!cloudflared --version

Selecting previously unselected package cloudflared.
(Reading database ... 122600 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.8.3) ...
Setting up cloudflared (2026.8.3) ...
Processing triggers for man-db (2.10.2-1) ...
cloudflared version 2026.8.3 (built 2026-08-31-10:04 UTC)


In [9]:
# Checking the server is alive or not

!curl http://127.0.0.1:11434/api/version

{"version":"0.33.2"}

## Tunneling the ollama server through cloudflare

In [10]:
import subprocess
import time
import re
import os

tunnel_process = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--url",
        "http://127.0.0.1:11434",
        "--http-host-header",
        "localhost:11434",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

tunnel_url = None

for _ in range(30):
    line = tunnel_process.stdout.readline()

    if line:
        print(line, end="")

        match = re.search(
            r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
            line
        )

        if match:
            tunnel_url = match.group(0)
            break

    time.sleep(0.2)

print("Tunnel:", tunnel_url)

os.environ["TUNNEL_URL"] = tunnel_url

2026-09-03T10:41:29Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-03T10:41:29Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-03T10:41:36Z INF +--------------------------------------------------------------------------------------------+
2026-09-03T10:41:36Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-03T10:41:36Z INF |  https://podcasts-biodiversity-idea-glance.trycloudfla

In [11]:
for _ in range(20):
    line = tunnel_process.stdout.readline()

    if line:
        print(line, end="")

2026-09-03T10:41:36Z INF +--------------------------------------------------------------------------------------------+
2026-09-03T10:41:36Z INF Cannot determine default configuration path. No file [config.yml config.yaml] in [~/.cloudflared ~/.cloudflare-warp ~/cloudflare-warp /etc/cloudflared /usr/local/etc/cloudflared]
2026-09-03T10:41:36Z INF Version 2026.8.3 (Checksum f29324fe934d1e100617484c78deef803c4dc2cd351d645bbde42e96b4fccc5e)
2026-09-03T10:41:36Z INF GOOS: linux, GOVersion: go1.26.4, GoArch: amd64
2026-09-03T10:41:36Z INF Settings: map[ha-connections:1 http-host-header:localhost:11434 protocol:quic url:http://127.0.0.1:11434]
2026-09-03T10:41:36Z INF cloudflared will not automatically update if installed by a package manager.
2026-09-03T10:41:36Z INF Generated Connector ID: 9913fd83-7c36-4581-af92-398c453912ef
2026-09-03T10:41:36Z INF Initial protocol quic
2026-09-03T10:41:36Z INF ICMP proxy will use 172.28.0.12 as source for IPv4
2026-09-03T10:41:36Z INF ICMP proxy will us

In [12]:
!echo "=== Runtime ==="
!uptime

!echo "=== Ollama process ==="
!ps aux | grep '[o]llama'

!echo "=== Port 11434 ==="
!ss -ltnp | grep 11434 || true

!echo "=== Last Ollama logs ==="
!tail -50 /tmp/ollama.log

=== Runtime ===
 10:41:42 up 9 min,  0 users,  load average: 1.60, 1.76, 0.87
=== Ollama process ===
root        2228  0.0  0.0   7376  3492 ?        S    10:38   0:00 /bin/bash /tmp/ollama_supervisor.sh
root        2231 16.9  0.4 2477208 57876 ?       Sl   10:38   0:36 ollama serve
root        2710 83.9  8.7 46318112 1162476 ?    Sl   10:39   1:57 /usr/local/lib/ollama/llama-server --model /root/.ollama/models/blobs/sha256-a3de86cd1c132c822487ededd47a324c50491393e6565cd14bafa40d0b8e686f --port 42733 --host 127.0.0.1 --no-webui --offline -c 4096 -np 1 --log-verbosity 4 --no-log-prefix --no-log-timestamps --no-jinja --chat-template chatml --load-mode none --flash-attn auto -b 512 -ub 512 --context-shift --keep 4
=== Port 11434 ===
LISTEN 0      4096       127.0.0.1:11434      0.0.0.0:*    users:(("ollama",pid=2231,fd=4))       
=== Last Ollama logs ===
slot get_availabl: id  0 | task -1 | selected slot by LRU, t_last = -1
srv  get_availabl: updating prompt cache
srv          load:  - lo

In [13]:
!tail -30 /tmp/ollama_supervisor.log

Thu Sep  3 10:38:08 AM UTC 2026: Starting Ollama...


In [14]:
!curl -i $TUNNEL_URL/api/version

HTTP/2 200 
date: Thu, 03 Sep 2026 10:41:43 GMT
content-type: application/json; charset=utf-8
content-length: 20
cf-ray: a35424c3dd15d95a-SIN
cf-cache-status: DYNAMIC
server: cloudflare

{"version":"0.33.2"}

In [15]:
!curl -i $TUNNEL_URL/api/version

!curl -i \
  -H "Origin: $TUNNEL_URL" \
  $TUNNEL_URL/api/version

HTTP/2 200 
date: Thu, 03 Sep 2026 10:41:43 GMT
content-type: application/json; charset=utf-8
content-length: 20
cf-ray: a35424c83ca3de08-SIN
cf-cache-status: DYNAMIC
server: cloudflare

{"version":"0.33.2"}HTTP/2 200 
date: Thu, 03 Sep 2026 10:41:43 GMT
content-type: application/json; charset=utf-8
content-length: 20
cf-ray: a35424c99e41f847-SIN
cf-cache-status: DYNAMIC
access-control-allow-origin: *
server: cloudflare

{"version":"0.33.2"}

In [17]:
!tail -30 /tmp/ollama.log

slot   operator(): id  0 | task 1491 | cached n_tokens = 1542, memory_seq_rm [1542, end)
slot   operator(): id  0 | task 1491 | cached n_tokens = 2054, memory_seq_rm [2054, end)
slot init_sampler: id  0 | task 1491 | init sampler, took 0.44 ms, tokens: text = 2483, total = 2483
slot print_timing: id  0 | task 1491 | n_gen =    100, tg =  29.39 t/s, tg_3s =  29.69 t/s
slot print_timing: id  0 | task 1491 | n_gen =    189, tg =  29.41 t/s, tg_3s =  29.43 t/s
slot print_timing: id  0 | task 1491 | n_gen =    275, tg =  29.06 t/s, tg_3s =  28.34 t/s
slot print_timing: id  0 | task 1491 | n_gen =    361, tg =  28.94 t/s, tg_3s =  28.56 t/s
srv  server_strea: conv_id= (empty=1)
srv          stop: cancel task, id_task = 1491
slot      release: id  0 | task 1491 | stop processing: n_tokens = 2854, truncated = 0
slot get_availabl: id  0 | task -1 |  - checking sim = 0.871 (2484/2851) > 0.100
slot get_availabl: id  0 | task -1 | selected slot by LCP similarity, f_sim_best = 0.871 (> 0.100 thold)